# upload_to_drive.py

In [ ]:
# pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
import os
import time
from pathlib import Path
import mimetypes

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload


# Your Google Drive folder ID from the link:
# https://drive.google.com/drive/folders/1WNB8eRnaFfQCzn4PgSAEoygGyoZzu414
DRIVE_FOLDER_ID = "1WNB8eRnaFfQCzn4PgSAEoygGyoZzu414"

SCOPES = ["https://www.googleapis.com/auth/drive.file"]


def get_drive_service():
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Put credentials.json in the same folder as this script
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json",
                SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("drive", "v3", credentials=creds)


def upload_file_to_drive(local_file_path):
    service = get_drive_service()

    local_file_path = Path(local_file_path)

    if not local_file_path.exists():
        raise FileNotFoundError(f"File not found: {local_file_path}")

    mime_type, _ = mimetypes.guess_type(local_file_path)
    if mime_type is None:
        mime_type = "application/octet-stream"

    file_metadata = {
        "name": local_file_path.name,
        "parents": [DRIVE_FOLDER_ID]
    }

    media = MediaFileUpload(
        str(local_file_path),
        mimetype=mime_type,
        resumable=True
    )

    start_time = time.perf_counter()

    uploaded_file = service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id, name, webViewLink, webContentLink"
    ).execute()

    upload_latency_ms = (time.perf_counter() - start_time) * 1000

    return {
        "file_id": uploaded_file.get("id"),
        "name": uploaded_file.get("name"),
        "webViewLink": uploaded_file.get("webViewLink"),
        "webContentLink": uploaded_file.get("webContentLink"),
        "upload_latency_ms": round(upload_latency_ms, 2)
    }


if __name__ == "__main__":
    result = upload_file_to_drive("capture.jpg")  # change this file name

    print("Uploaded successfully")
    print("File name:", result["name"])
    print("File ID:", result["file_id"])
    print("View link:", result["webViewLink"])
    print("Download link:", result["webContentLink"])
    print("Upload latency:", result["upload_latency_ms"], "ms")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=658174009669-oir0rra4dhpl7b0v18nct5ma5k9ij8k5.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A63766%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=pBOYIHmI4olLLvdsvlqkerGFcehpm0&code_challenge=2MKq6T1U9oHj3OzieQ-qMHNfHk5rpi0jEzArRtX160k&code_challenge_method=S256&access_type=offline
